## Importação da bibliotecas utilizadas.


- `pandas (pd)`: Biblioteca para manipulação e análise de dados em tabelas (DataFrames).  
- `re`: Biblioteca para trabalhar com expressões regulares (buscas e substituições em texto).  
- `dateutil.relativedelta`: Permite fazer cálculos com datas, adicionando ou subtraindo períodos.  
- `datetime`: Módulo padrão para trabalhar com datas e horários.  
- `boto3`: Biblioteca oficial da AWS para acessar serviços como S3, EC2, etc.  
- `botocore`: Biblioteca base do boto3, usada para tratamento de erros e configurações detalhadas.  
- `StringIO`: Permite tratar strings como arquivos, útil para manipular dados em memória.  
- `os`: Biblioteca padrão para interagir com o sistema operacional, como variáveis de ambiente.  
- `load_dotenv` (de `dotenv`): Função que carrega variáveis do arquivo `.env` para o ambiente do sistema.


In [77]:
import pandas as pd
import re
from dateutil.relativedelta import relativedelta
from datetime import datetime
import boto3
import botocore
from io import StringIO
import os
from dotenv import load_dotenv

## Definindo credenciais de acesso na plataforma AWS e iniciando uma sessão

Carregar variáveis de ambiente de um arquivo .env

In [78]:
load_dotenv()

True

Atribuição das variaveis de acesso a partir da extração das variaveis de ambiente

In [79]:
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_session_token = os.getenv("AWS_SESSION_TOKEN")
region = os.getenv("AWS_REGION")

Cria uma sessão AWS usando as credenciais anteriores

In [80]:
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token,
    region_name=region
)


s3 = session.client('s3')

## Leitura do CSV

Define o nome do bucket S3 onde os arquivos serão salvos e os nomes dos arquivos CSV. 
O arquivo principal com os dados originais e três arquivos resultantes das análises.

In [81]:
bucket_nome = "rhamon-pires-sprint04-bucket"
base_dados_path = "DEVEDORES_DIVIDA_ATIVA-2024-12.csv" 
analise_1_path = "analise_1.csv" 
analise_2_path = "analise_2.csv"
analise_3_path = "analise_3.csv"

Busca o arquivo CSV principal dentro do bucket S3, usando o nome do bucket e o caminho (key) do arquivo.

In [82]:
obj = s3.get_object(Bucket=bucket_nome, Key=base_dados_path)

Lê o conteúdo do arquivo baixado do S3 e o decodifica usando a codificação 'latin1', já que o CSV está nesse formato.

In [83]:
csv_content = obj['Body'].read().decode('latin1')


Carrega o conteúdo CSV em uma tabela (DataFrame) do pandas, usando ponto e vírgula (`;`) como separador.

In [84]:
df = pd.read_csv(StringIO(csv_content),  sep=';')


Exibir as primeiras 5 linhas do DataFrame

In [85]:
df.head(5)

,DATA DE GERAÇÃO,CPF/CNPJ DEVEDOR,TIPO DE PESSOA,TIPO DO DEVEDOR,NOME DO DEVEDOR,NÚMERO DO TERMO DE INSCRIÇÃO,DATA DA INSCRIÇÃO,SEQUENCIAL DO CRÉDITO,SITUAÇÃO DO CRÉDITO,MOTIVO DA SUSPENSÃO,ORIGEM,CREDOR,SALDO DEVEDOR SEM HONORÁRIOS,OBSERVAÇÃO
0,31/12/2024,XXX.247.618-XX,Pessoa física,Principal,Jose Roberto Sevieri,2023.001-058,17/03/2023 18:16,CR2023/0000123,Parcelado,NaN,Multa,Banco Central,"R$ 30.625,75",NaN
1,31/12/2024,XXX.476.009-XX,Pessoa física,Principal,Aldo de Almeida Junior,0084/2010,17/08/2010 00:00,CR2013/0005711,Parcelado,NaN,Multa,Banco Central,"R$ 27.089,25",NaN
2,31/12/2024,XXX.579.970-XX,Pessoa física,Principal,Marco Antonio de Mattos Filho,2024.001-162,26/04/2024 07:37,CR2024/0000225,Parcelado,NaN,Multa,Banco Central,"R$ 29.847,88",NaN
3,31/12/2024,XXX.188.836-XX,Pessoa física,Principal,Leandro Martins,2023.001-117,30/06/2023 18:00,CR2023/0000282,Parcelado,NaN,Multa,Banco Central,"R$ 22.262,44",NaN
4,31/12/2024,00.689.364/0001-20,Pessoa jurídica,Principal,Bok Administracao e Participacoes S/A,0203/2008,04/09/2008 00:00,CR2013/0005379,Parcelado,NaN,Multa,Banco Central,"R$ 35.461,23",NaN


## Tratamento e formatação dos dados

Converte as colunas para tipos adequados (datas e números) para garantir que os dados possam ser manipulados corretamente.  
Remove caracteres indesejados (R$)e ajusta formatos para evitar erros nas operações matemáticas e análises futuras.

In [86]:
# Supondo df já carregado e com as colunas apropriadas
df['DATA DA INSCRIÇÃO'] = pd.to_datetime(df['DATA DA INSCRIÇÃO'], dayfirst=True, errors='coerce')

df['SALDO DEVEDOR SEM HONORÁRIOS'] = pd.to_numeric(
    df['SALDO DEVEDOR SEM HONORÁRIOS']
      .str.replace(r'[R$\s.]', '', regex=True)  # Remove "R$", espaços e pontos
      .str.replace(',', '.', regex=False),      # Troca vírgula por ponto decimal
    errors='coerce'
)



In [87]:
df.head(3)

,DATA DE GERAÇÃO,CPF/CNPJ DEVEDOR,TIPO DE PESSOA,TIPO DO DEVEDOR,NOME DO DEVEDOR,NÚMERO DO TERMO DE INSCRIÇÃO,DATA DA INSCRIÇÃO,SEQUENCIAL DO CRÉDITO,SITUAÇÃO DO CRÉDITO,MOTIVO DA SUSPENSÃO,ORIGEM,CREDOR,SALDO DEVEDOR SEM HONORÁRIOS,OBSERVAÇÃO
0,31/12/2024,XXX.247.618-XX,Pessoa física,Principal,Jose Roberto Sevieri,2023.001-058,2023-03-17 18:16:00,CR2023/0000123,Parcelado,NaN,Multa,Banco Central,"30,625.75",NaN
1,31/12/2024,XXX.476.009-XX,Pessoa física,Principal,Aldo de Almeida Junior,0084/2010,2010-08-17 00:00:00,CR2013/0005711,Parcelado,NaN,Multa,Banco Central,"27,089.25",NaN
2,31/12/2024,XXX.579.970-XX,Pessoa física,Principal,Marco Antonio de Mattos Filho,2024.001-162,2024-04-26 07:37:00,CR2024/0000225,Parcelado,NaN,Multa,Banco Central,"29,847.88",NaN


## Manipulação dos dados para análises

As 3 análises principais a serem feitas:

**1. Entre pessoas físicas e jurídicas, qual grupo concentra a maior proporção da dívida total e qual é a situação predominante dessas dívidas?**

**2. Como a quantidade e o valor total das dívidas em execução variam conforme o tempo de atraso, e quais faixas de atraso concentram a maior parte da dívida ativa?**

**3. Qual é a distribuição da quantidade e do valor das dívidas <u>milionárias</u> entre os tipos de empresa LTDA, S/A e Outros, e como o tempo médio de atraso varia entre esses grupos.**


Função para calcular quantos anos se passaram desde uma data até hoje.  
Retorna um inteiro ou `None`, caso a data seja inválida ou nula.

In [88]:
def calcular_anos_atraso(data_inscricao):

    if pd.isnull(data_inscricao):
        return None
    
    hoje = datetime.now()
    diff = relativedelta(hoje, data_inscricao)

    return diff.years

### ANÁLISE 1 

**Entre pessoas físicas e jurídicas, qual grupo concentra a maior proporção da dívida total e qual é a situação predominante dessas dívidas?**

Agrupa os dados por tipo de pessoa, calculando o total da dívida e a quantidade de registros.  
Calcula os totais gerais e as porcentagens correspondentes para cada grupo.  
Soma a dívida por tipo de pessoa e situação do crédito, calculando a participação percentual de cada situação.  
Por fim, junta tudo em um único df_intervalos , organizando as colunas para facilitar a análise.

In [89]:
# df_intervalos  geral por tipo de pessoa
df_filtrado = df.groupby('TIPO DE PESSOA').agg(
    TOTAL_DIVIDA=('SALDO DEVEDOR SEM HONORÁRIOS', 'sum'),
    QUANTIDADE=('TIPO DE PESSOA', 'count')
).copy()

# Totais 
total_divida_geral = df_filtrado['TOTAL_DIVIDA'].sum()
total_quantidade_geral = df_filtrado['QUANTIDADE'].sum()

# Porcentagens
df_filtrado['% TOTAL_DIVIDA'] = (df_filtrado['TOTAL_DIVIDA'] / total_divida_geral) * 100
df_filtrado['% QUANTIDADE'] = (df_filtrado['QUANTIDADE'] / total_quantidade_geral) * 100

#soma por situação
valores_situacoes = (
    df.groupby(['TIPO DE PESSOA', 'SITUAÇÃO DO CRÉDITO'])['SALDO DEVEDOR SEM HONORÁRIOS']
    .sum()
    .unstack(fill_value=0)
)

# Porcentagem por situação
percentuais_situacoes = valores_situacoes.div(valores_situacoes.sum(axis=1), axis=0) * 100
percentuais_situacoes.columns = [f'% {col.upper()}' for col in percentuais_situacoes.columns]

# Juntar ao df_intervalos 
df_filtrado = df_filtrado.join(percentuais_situacoes)

# Reordena colunas
colunas_ordem = ['QUANTIDADE', '% QUANTIDADE', 'TOTAL_DIVIDA', '% TOTAL_DIVIDA'] + list(percentuais_situacoes.columns)
df_filtrado = df_filtrado[colunas_ordem]

Configura o pandas para mostrar números decimais com duas casas e separador de milhares.  
Exibe a tabela final formatada (`resultado_final`).

In [90]:
pd.set_option('display.float_format', '{:,.2f}'.format)
df_filtrado

,QUANTIDADE,% QUANTIDADE,TOTAL_DIVIDA,% TOTAL_DIVIDA,% EM EXECUÇÃO,% NÃO EXECUTADO,% PARCELADO,% SUSPENSO
TIPO DE PESSOA,,,,,,,,
CPF/CNPJ não encontrado no processo,3,0.12,"947,432.34",0.00,100.00,0.00,0.00,0.00
Pessoa física,907,36.54,"3,991,867,983.06",20.35,96.40,0.17,0.12,3.30
Pessoa jurídica,1572,63.34,"15,618,983,426.99",79.64,86.80,0.16,0.56,12.48


Função para enviar um DataFrame para um bucket S3 como arquivo CSV.  
Antes de enviar, verifica se o arquivo já existe no bucket; se existir, ignora o upload.  
Caso contrário, converte o DataFrame para CSV em memória e faz o upload para o S3.

In [91]:
def upload_df_para_s3(df, bucket_nome, arquivo_nome):
    s3 = boto3.client('s3')
    try:
        # Verifica se o arquivo já existe no bucket
        s3.head_object(Bucket=bucket_nome, Key=arquivo_nome)
        print(f"O arquivo '{arquivo_nome}' já existe no bucket '{bucket_nome}'. Upload ignorado.")

    except botocore.exceptions.ClientError as e:

        if e.response['Error']['Code'] == '404':

            csv_buffer = StringIO()
            df.to_csv(csv_buffer, index=False)
            csv_buffer.seek(0)

            s3.put_object(Bucket=bucket_nome, Key=arquivo_nome, Body=csv_buffer.getvalue())
            print(f"Arquivo '{arquivo_nome}' enviado para 's3://{bucket_nome}/{arquivo_nome}' com sucesso.")
            
        else:
            raise

Após a definição, é feito a chamada para realizar a tentativa de upload.

In [ ]:
upload_df_para_s3(df_filtrado, bucket_nome, analise_1_path)

Salva o DataFrame como CSV local, sem incluir índice.

In [93]:
df_filtrado.to_csv(analise_1_path, index=False)

### ANÁLISE 2

**Como a quantidade e o valor total das dívidas em execução variam conforme o tempo de atraso, e quais faixas de atraso concentram a maior parte da dívida ativa?**

Filtra os dados para pessoas jurídicas com situação "Em execução".  
Calcula os anos de atraso e classifica em faixas predefinidas.  
Agrupa por faixa, calculando quantidade, média e total da dívida, além das porcentagens relativas.  
Por fim, arredonda os valores para melhor visualização.

In [94]:
# Filtra apenas pessoas jurídicas com situação "Em execução"
df_filtrado = df[
    (df['TIPO DE PESSOA'] == 'Pessoa jurídica') &
    (df['SITUAÇÃO DO CRÉDITO'] == 'Em execução')
].copy()

# Define as faixas de tempo de atraso em anos
bins = [0, 1, 3, 5, 10, 15, 20, 25, float('inf')]
labels = ['0 - 1A', '1A - 3A', '3A - 5A', '5A - 10A', '10A - 15A', '15A - 20A', '20A - 25A', '25A+']

# Classifica cada dívida na faixa correspondente de atraso
df_filtrado['INTERVALO_ATRASO'] = pd.cut(
    df_filtrado['DATA DA INSCRIÇÃO'].apply(calcular_anos_atraso),
    bins=bins,
    labels=labels,
    right=False
)

# Agrupa os dados por faixa de atraso e calcula métricas
df_intervalos = df_filtrado.groupby('INTERVALO_ATRASO', observed=False).agg(
    QUANTIDADE=('SALDO DEVEDOR SEM HONORÁRIOS', 'count'),
    MEDIA_DIVIDA=('SALDO DEVEDOR SEM HONORÁRIOS', 'mean'),
    TOTAL_DIVIDA=('SALDO DEVEDOR SEM HONORÁRIOS', 'sum')
).reset_index()

# Calcula totais gerais para as porcentagens
total_dividas = df_filtrado['SALDO DEVEDOR SEM HONORÁRIOS'].sum()
quantidade_total = df_intervalos['QUANTIDADE'].sum()

# Adiciona colunas de porcentagem sobre o total
df_intervalos['% QUANTIDADE'] = (df_intervalos['QUANTIDADE'] / quantidade_total) * 100
df_intervalos['% TOTAL_DIVIDA'] = (df_intervalos['TOTAL_DIVIDA'] / total_dividas) * 100

# Arredonda os valores para facilitar a visualização
df_intervalos['MEDIA_DIVIDA'] = df_intervalos['MEDIA_DIVIDA'].round(2)
df_intervalos['TOTAL_DIVIDA'] = df_intervalos['TOTAL_DIVIDA'].round(2)
df_intervalos['% TOTAL_DIVIDA'] = df_intervalos['% TOTAL_DIVIDA'].round(2)

# Reorganiza as colunas para apresentação final
df_intervalos = df_intervalos[['INTERVALO_ATRASO', 'QUANTIDADE', '% QUANTIDADE', 'TOTAL_DIVIDA', '% TOTAL_DIVIDA', 'MEDIA_DIVIDA']]


In [95]:
df_intervalos

,INTERVALO_ATRASO,QUANTIDADE,% QUANTIDADE,TOTAL_DIVIDA,% TOTAL_DIVIDA,MEDIA_DIVIDA
0,0 - 1A,3,0.23,"99,495.84",0.00,"33,165.28"
1,1A - 3A,61,4.65,"128,060,134.04",0.94,"2,099,346.46"
2,3A - 5A,106,8.08,"263,112,229.58",1.94,"2,482,190.85"
3,5A - 10A,189,14.41,"801,830,457.36",5.91,"4,242,489.19"
4,10A - 15A,239,18.22,"1,489,195,283.50",10.98,"6,230,942.61"
5,15A - 20A,290,22.10,"3,514,924,893.31",25.93,"12,120,430.67"
6,20A - 25A,340,25.91,"6,051,378,544.88",44.63,"17,798,172.19"
7,25A+,84,6.40,"1,309,125,080.45",9.66,"15,584,822.39"


Faz o upload (se for possível), e exporta o dataframe para um CSV local

In [ ]:
upload_df_para_s3(df_intervalos , bucket_nome, analise_2_path)

df_intervalos .to_csv(analise_2_path, index=False)

A função usa expressões regulares para identificar variações comuns de “S/A” e “LTDA” no nome da empresa,  
considerando que podem aparecer com espaços ou símbolos entre as letras (como pontos, barras ou hífens).  
Isso garante que formatos diferentes sejam reconhecidos corretamente.

In [97]:
def classificar_tipo_empresa(nome):
    if pd.isna(nome):
        return 'OUTROS'
    nome_upper = nome.upper()
    regex_sa = r'\bS\s*[\./\\\-]?\s*A\b\.?'
    regex_ltda = r'\bL\s*[\./\\\-]?\s*T\s*[\./\\\-]?\s*D\s*[\./\\\-]?\s*A\b\.?'

    if re.search(regex_sa, nome_upper):
        return 'S/A'
    elif re.search(regex_ltda, nome_upper):
        return 'LTDA'
    else:
        return 'OUTROS'

### ANÁLISE 3

**Qual é a distribuição da quantidade e do valor das dívidas entre os tipos de empresa LTDA, S/A e Outros, e como o tempo médio de atraso varia entre esses grupos.**

Filtra os dados para pessoas jurídicas com dívida maior que 1 milhão e situação ativa (Não suspenso).  
Classifica o tipo da empresa (S/A, LTDA, Outros) e converte datas para cálculo do tempo de atraso.  
Agrupa por tipo de empresa calculando quantidade, soma, média da dívida e média do atraso.  
Calcula porcentagens relativas, arredonda valores e formata o atraso em anos, meses e dias.  
Por fim, ordena e organiza os dados para melhor visualização.

In [98]:
# Filtrar apenas Pessoa Jurídica
df_pj = df[
    (df['TIPO DE PESSOA'] == 'Pessoa jurídica') & 
    ~(df['SITUAÇÃO DO CRÉDITO'] == 'Suspenso') &
    (df['SALDO DEVEDOR SEM HONORÁRIOS'] >= 1_000_000)
].copy()

# Classificar os tipos (S/A, LTDA, OUTROS)
df_pj['TIPO_EMPRESA'] = df_pj['NOME DO DEVEDOR'].apply(classificar_tipo_empresa)

# Converter datas
df_pj['DATA DA INSCRIÇÃO'] = pd.to_datetime(df_pj['DATA DA INSCRIÇÃO'], errors='coerce')

# Calcular tempo de atraso em anos decimais
df_pj['TEMPO_ATRASO_ANOS'] = (
    (datetime.now() - df_pj['DATA DA INSCRIÇÃO']).dt.days / 365.0
)

# Agrupar e calcular métricas
df_intervalos  = df_pj.groupby('TIPO_EMPRESA').agg(
    QUANTIDADE=('NOME DO DEVEDOR', 'count'),
    TOTAL_DIVIDA=('SALDO DEVEDOR SEM HONORÁRIOS', 'sum'),
    MEDIA_DIVIDA=('SALDO DEVEDOR SEM HONORÁRIOS', 'mean'),
    MEDIA_ATRASO_ANOS=('TEMPO_ATRASO_ANOS', 'mean')
).reset_index()

# Calcular totais para porcentagens
total_qtd = df_intervalos ['QUANTIDADE'].sum()
total_div = df_intervalos ['TOTAL_DIVIDA'].sum()

df_intervalos ['% QUANTIDADE'] = (df_intervalos ['QUANTIDADE'] / total_qtd) * 100
df_intervalos ['% TOTAL_DIVIDA'] = (df_intervalos ['TOTAL_DIVIDA'] / total_div) * 100

# Arredondar valores numéricos (menos atraso que vamos formatar)
df_intervalos  = df_intervalos .round({
    'TOTAL_DIVIDA': 2,
    'MEDIA_DIVIDA': 2,
    '% QUANTIDADE': 2,
    '% TOTAL_DIVIDA': 2
})

# Ordenar na ordem desejada
ordem = ['LTDA', 'S/A', 'OUTROS']
df_intervalos ['TIPO_EMPRESA'] = pd.Categorical(df_intervalos ['TIPO_EMPRESA'], categories=ordem, ordered=True)
df_intervalos  = df_intervalos .sort_values('TIPO_EMPRESA').reset_index(drop=True)

# Reordenar colunas para visualização
df_intervalos  = df_intervalos [['TIPO_EMPRESA', 'QUANTIDADE', '% QUANTIDADE', 'TOTAL_DIVIDA',
                 '% TOTAL_DIVIDA', 'MEDIA_DIVIDA', 'MEDIA_ATRASO_ANOS']]


In [99]:
df_intervalos 

,TIPO_EMPRESA,QUANTIDADE,% QUANTIDADE,TOTAL_DIVIDA,% TOTAL_DIVIDA,MEDIA_DIVIDA,MEDIA_ATRASO_ANOS
0,LTDA,449,74.83,"7,997,126,867.41",59.44,"17,810,972.98",15.58
1,S/A,81,13.50,"4,690,899,858.55",34.87,"57,912,343.93",17.49
2,OUTROS,70,11.67,"765,340,406.23",5.69,"10,933,434.37",15.11


Faz o upload(se for possível), e exporta o dataframe para um CSV local

In [ ]:
upload_df_para_s3(df_intervalos , bucket_nome, analise_3_path)

df_intervalos .to_csv(analise_3_path, index=False)

Para garantir que os arquivos CSV enviados para o bucket S3 estejam corretos, o código lê cada um deles, carrega em DataFrames e exibe as primeiras 3 linhas para conferência rápida dos dados.

In [101]:
csv_leitura_lista = ["analise_1.csv", "analise_2.csv", "analise_3.csv"]

for arquivo_nome in csv_leitura_lista:

    obj = s3.get_object(Bucket=bucket_nome, Key=arquivo_nome)
    content = obj['Body'].read().decode('utf-8')
    df = pd.read_csv(StringIO(content))

    print(f"Conteúdo do arquivo {arquivo_nome}:\n")
    print(df.head(3))
    print("\n" + "="*50 + "\n")



Conteúdo do arquivo analise_1.csv:

   QUANTIDADE  % QUANTIDADE      TOTAL_DIVIDA  % TOTAL_DIVIDA  % EM EXECUÇÃO  \
0           3          0.12        947,432.34            0.00         100.00   
1         907         36.54  3,991,867,983.06           20.35          96.40   
2        1572         63.34 15,618,983,426.99           79.64          86.80   

   % NÃO EXECUTADO  % PARCELADO  % SUSPENSO  
0             0.00         0.00        0.00  
1             0.17         0.12        3.30  
2             0.16         0.56       12.48  


Conteúdo do arquivo analise_2.csv:

  FAIXA_DIVIDA  QUANTIDADE  % QUANTIDADE   TOTAL_DIVIDA  % TOTAL_DIVIDA  \
0       0 - 1A           3          0.23      99,495.84            0.00   
1      1A - 3A          61          4.65 128,060,134.04            0.94   
2      3A - 5A         106          8.08 263,112,229.58            1.94   

   MEDIA_DIVIDA  
0     33,165.28  
1  2,099,346.46  
2  2,482,190.85  


Conteúdo do arquivo analise_3.csv:

  TIPO_EMP